# 체간 기울기에서 어깨와 골반을 한 점씩 바꾸기

23번 실험에서 사람이 두 번 모두 더 기울었다고 읽었지만 모델은 더 세워졌다고 읽은 a05와 b02를 봅니다. **같은 관절을 이륙과 무릎 통과 양쪽 프레임에서** 수동점으로 교체합니다. 원본 영상, 프레임, 수행자, 가까운 쪽, 다른 관절은 고정합니다. 1차와 2차 수동점은 섞지 않습니다.

**Run All**로 실행하세요. 23번 실험에서 저장한 좌표를 사용하므로 모델 추론이나 학습이 필요하지 않습니다.

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display

from wlcv.config import ROOT, VIDEOS
from wlcv.evaluation.trunk_joint_swap import (
    changes_by_condition, plot_changes, plot_overlays, swap_angles,
)

source = ROOT / "outputs/pose_baseline/trunk_model"
frames = pd.read_csv(source / "frames.csv")
joints = pd.read_csv(source / "joints.csv")
metadata = pd.read_csv(ROOT / "data/metadata/videos.csv").set_index("video_id")
angles = swap_angles(frames, joints)
change = changes_by_condition(angles)

## 두 시점 사이의 변화 방향 확인

각도는 골반에서 어깨로 향하는 선과 화면 위쪽 수직선 사이의 각도입니다. 표는 **무릎 통과 각도 − 이륙 각도**입니다. 음수는 몸통이 더 세워짐, 양수는 더 기울어짐을 뜻합니다. ‘두 점 수동’은 해당 회차에서 사람이 두 점을 찍었을 때의 비교값입니다. 표에서 모델과 부호가 달라지는 조건을 찾고, 두 판독에서 같은 결과가 나오는지 확인하세요.

In [ ]:
display(change.style.format(precision=2))
chart = plot_changes(change)
plt.show()

## 같은 원본에서 교체한 점 확인

열은 각각 어깨만 교체, 골반만 교체이고, 위 행은 이륙, 아래 행은 무릎 통과입니다. 흰 점선은 원래 모델선, 청록색은 한 점 교체 후의 선, 노란 마름모는 바꾼 수동점입니다. 아래에서 영상과 회차를 바꿔 볼 수 있습니다. 각도는 사진을 확대하기 전의 원본 좌표로 계산합니다.

In [ ]:
video = "b02_sn_h_02"  # 다른 영상: "a05_sn_f_01"
pass_number = 1        # 1 또는 2
overlay = plot_overlays(angles, video, pass_number, VIDEOS / metadata.at[video, "filename"])
plt.show()
selection = (angles["영상"] == video) & (angles["판독 회차"] == pass_number)
display(angles.loc[selection, ["시점", "프레임", "조건", "각도 (°)", "수동 골반 판독"]]
        .style.format(precision=2))

## 결과 저장

네 조건의 각 프레임 각도·좌표, 선수별 변화량, 요약 그래프와 두 회차의 원본 비교 그림을 로컬에 저장합니다. 수동 골반은 소켓 위치를 추정한 점입니다. 이 실험으로 확인하는 것은 좌표를 바꾸었을 때 계산된 각도의 변화입니다.

In [ ]:
output = ROOT / "outputs/pose_baseline/trunk_joint_swap"
output.mkdir(parents=True, exist_ok=True)
angles.to_csv(output / "angles.csv", index=False)
change.to_csv(output / "changes.csv", index=False)
chart.savefig(output / "changes.png", dpi=135)
for (video_id, n), group in angles.groupby(["영상", "판독 회차"], sort=False):
    image = plot_overlays(group, video_id, n, VIDEOS / metadata.at[video_id, "filename"])
    image.savefig(output / f"{video_id}_pass{n}.png", dpi=135)
    plt.close(image)
print(output)